In [1]:
import sys

sys.path.append("..")

In [2]:
import os
from src.api.dependencies.injectables import (
    get_mongo_vdb,
    get_chat_model,
    get_topic_selector,
    get_rag_engine,
    get_topic_prompt_builder,
    get_embedding_model,
    get_agent,
    get_election_searcher,
)
from src.mongo import get_mongo_db
from src import ENV
from src.agent.schemas import Platform, Topic
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_core.chat_history import InMemoryChatMessageHistory
from IPython.display import display, Markdown
from langchain_core.messages import AIMessage, HumanMessage


In [3]:
chat_model = get_chat_model()
emb_model = get_embedding_model()
topic_selector = get_topic_selector()
db = get_mongo_db()
vector_db = get_mongo_vdb(emb_model, db)
rag_engine = get_rag_engine(vector_db)
search_election = get_election_searcher(chat_model, vector_db)
topic_prompt_builder = get_topic_prompt_builder(db, vector_db, search_election)

In [4]:
agent = get_agent(
    chat_model=chat_model,
    classify_topic=topic_selector,
    rag_retrieve=rag_engine,
    build_topic_prompts=topic_prompt_builder,
)  # type: ignore

In [5]:
history = []
query = ""
result = ""

In [7]:
if query and result:
    history.extend([HumanMessage(query), AIMessage(result)])
query = "que dice el sobre la salud?"
result = ""
async for token in agent.stream([*history, HumanMessage(content=query)]):
    if token.type == "info" or token.type == "error":
        display(Markdown(token.content), clear=True)
        continue
    result += token.content
    display(Markdown(result), clear=True)
# display(Markdown(result), clear=False)

Según la información encontrada, el plan de gobierno de Jorge "Tuto" Quiroga, de la Alianza Libertad y Democracia (LIBRE), incluye varias propuestas relacionadas con la salud:

- Fortalecer el marco normativo del sistema de salud, con penas más severas para actos de corrupción, robo de recursos médicos y negligencia administrativa, y crear un Instituto de Conciliación y Arbitraje especializado en conflictos sanitarios.
- Liberar aranceles de importación en medicamentos, insumos y equipos no producidos en Bolivia.
- Crear una base de datos nacional para monitorear y evaluar el desempeño de los establecimientos de salud, con gestión financiera transparente mediante contratos inteligentes (Blockchain) y una plataforma digital de acceso público para presupuestos y gastos hospitalarios.
- Descentralizar el sistema de salud transfiriendo la gestión operativa de hospitales y centros de salud a los gobiernos departamentales y municipales, eliminando organismos intermedios como AISEM y CEAS.
- Enfocarse en la prevención y promoción de la salud, con un modelo basado en la corresponsabilidad del Estado, sociedad civil y ciudadanos, y en atención integral con estándares internacionales.
- Humanizar la atención médica, aumentando el tiempo en la primera consulta, incorporando estudiantes de medicina en la gestión de pacientes, y promoviendo campañas de prevención de enfermedades crónicas.
- Implementar políticas de salud ocupacional para garantizar condiciones laborales seguras, promover entornos laborales saludables y apoyar a trabajadores con discapacidades adquiridas en el trabajo.

Para más detalles, puedes consultar el plan completo en [este enlace](https://drive.google.com/file/d/1WVEWSo9589x8u9lirA6mEMBhcFTghSeR/view).

In [7]:
%rm -rf chroma_db
%cp -r ../chroma_db .
